### This script extracts catchment attributes for each HydroBasin with available observed water quality data:

In [9]:
import pandas as pd
import numpy as np
from time import time 
from tqdm import tqdm
import pickle
import geopandas as gpd
import dask_geopandas
import fiona
import os

In [10]:
# define the output folder path:

output_folder = '../../output_data/CNP_data_catchments'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

#### Load file containing all aggregated obsereved water quality data: in order to obtain all HYBAS_ID's for that data are available

In [11]:
tic = time()
wq_data = pickle.load(open('../../output_data/aggregate_stations_by_subbasins/daily_wq_data_by_subbasin_0km_merged.pkl','rb'))
print("Elapsed %.2f seconds" % ((time() - tic)))

Elapsed 24.73 seconds


### Load BasinATLAS database containing hydro-environmental attributes for HydroBasins (finest spatial resolution level 12):

In [16]:
gdb_path = '../../input_data/geodata/HydroBasins/BasinATLAS_v10.gdb'


# name of feature-class within geodatabase
feature_class_name = 'BasinATLAS_v10_lev12'

# create complete path to Feature-Class
feature_class_path = f'{gdb_path}\\{feature_class_name}'



In [17]:
# use dask_geopandas to read in feature class of BasinATLAS_v10_lev12 

start = time()
layers = fiona.listlayers(gdb_path)
layers
selected_layer = layers[11]

# read in layer 'BasinATLAS_v10_lev12'
#subbasins_lev12 = gpd.GeoDataFrame.from_file(gdb_path,layer= 'BasinATLAS_v10_lev12')
data_level12 = dask_geopandas.read_file(gdb_path,layer= 'BasinATLAS_v10_lev12', npartitions = 4)
end = time()

runtime = (end - start)
print(runtime)

0.3500070571899414


In [19]:
# now put partitions together again: to obtain one large df:

start = time()
subbasins_lev12 = data_level12.compute()
end= time()
dauer = (end-start)
print(f'{dauer}sec')

175.08476281166077sec


### Now only filter the attributes for the Basins for which data are available in wq_data:


In [20]:
# create array containing all unique HYBAS-ID's for which data are available:
HYBAS_ava = wq_data['HYBAS_ID'].unique()

In [21]:
basin_atlas_v10_l12_data_available = subbasins_lev12[subbasins_lev12['HYBAS_ID'].isin(HYBAS_ava)]

In [22]:
basin_atlas_v10_l12_data_available.to_csv('../../output_data/CNP_data_catchments/basin_atlas_v10_l12_data_available.csv')

In [23]:
basin_atlas_v10_l12_data_available.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 29491 entries, 10142 to 1016699
Columns: 297 entries, HYBAS_ID to geometry
dtypes: float32(13), float64(12), geometry(1), int16(261), int32(10)
memory usage: 20.4 MB
